# TemporalRec — Event-Policy EDA

This notebook compares possible definitions of an **eligible interaction** before we build sequences. It reads the tie-free output of Notebook 01 and does not delete or overwrite data.

We are asking: should a next-item sequence use all behavior (`view`, `addtocart`, `transaction`) or only higher-intent actions?

## Before you run

Run `01_data_quality_cleaning.ipynb` successfully first. Then copy its `RUN_TAG` into `INPUT_RUN_TAG` below. Change `ANALYSIS_TAG` if you rerun this notebook so old reports remain untouched.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone

import pandas as pd

pd.set_option("display.max_columns", 30)

PROJECT_ROOT = Path.cwd()
INPUT_RUN_TAG = "notebook_run_2026_10_13"  # Must match Notebook 01.
ANALYSIS_TAG = "event_policy_eda_2026_10_13"

INPUT_FILE = PROJECT_ROOT / "data" / "interim" / INPUT_RUN_TAG / "events_clean_for_next_steps.csv"
REPORT_DIR = PROJECT_ROOT / "reports" / ANALYSIS_TAG

assert INPUT_FILE.exists(), f"Cannot find {INPUT_FILE}. Run Notebook 01 first or correct INPUT_RUN_TAG."
assert not REPORT_DIR.exists(), f"{REPORT_DIR} exists. Change ANALYSIS_TAG to preserve older results."
REPORT_DIR.mkdir(parents=True)

print(f"Input:  {INPUT_FILE}")
print(f"Reports: {REPORT_DIR}")

In [ ]:
EVENT_COLUMNS = ["timestamp", "visitorid", "event", "itemid", "transactionid"]
DTYPES = {
    "timestamp": "int64",
    "visitorid": "int64",
    "event": "category",
    "itemid": "int64",
    "transactionid": "Int64",
}
events = pd.read_csv(INPUT_FILE, dtype=DTYPES)
assert events.columns.tolist() == EVENT_COLUMNS
assert events.duplicated(EVENT_COLUMNS, keep=False).sum() == 0
assert events.duplicated(["visitorid", "timestamp"], keep=False).sum() == 0

print(f"Events available for policy analysis: {len(events):,}")
events.head()

## 1. Understand the three event types

First, count what each action means in the data. A `view` is weaker purchase intent than an `addtocart` or `transaction`, but it is also much more common.

In [ ]:
event_distribution = (
    events["event"].value_counts(dropna=False)
    .rename_axis("event")
    .reset_index(name="event_count")
)
event_distribution["share_of_events"] = event_distribution["event_count"] / len(events)
event_distribution.to_csv(REPORT_DIR / "01_event_distribution.csv", index=False)
event_distribution

## 2. Compare candidate event policies

For each policy we count visitors, events, items, and visitors with at least five events. This is still only a comparison—**we do not select a winner automatically**.

The minimum of five includes both history and future targets. We will later apply the stricter iterative user-and-item 5-core filter only after choosing the policy.

In [ ]:
POLICIES = {
    "all_events": ["view", "addtocart", "transaction"],
    "view_only": ["view"],
    "view_and_addtocart": ["view", "addtocart"],
    "cart_and_transaction": ["addtocart", "transaction"],
    "transaction_only": ["transaction"],
}
MIN_EVENTS_PER_USER = 5

comparison_rows = []
for policy_name, allowed_events in POLICIES.items():
    policy_events = events.loc[events["event"].isin(allowed_events)].copy()
    user_event_counts = policy_events.groupby("visitorid").size()
    active_users = user_event_counts[user_event_counts >= MIN_EVENTS_PER_USER].index
    active_events = policy_events.loc[policy_events["visitorid"].isin(active_users)]

    comparison_rows.append({
        "policy": policy_name,
        "eligible_events": " + ".join(allowed_events),
        "all_events": len(policy_events),
        "all_visitors": policy_events["visitorid"].nunique(),
        "all_items": policy_events["itemid"].nunique(),
        "visitors_with_at_least_5_events": len(active_users),
        "events_for_visitors_with_at_least_5": len(active_events),
        "items_after_user_minimum": active_events["itemid"].nunique(),
        "user_coverage_after_minimum": len(active_users) / policy_events["visitorid"].nunique(),
        "event_coverage_after_minimum": len(active_events) / len(policy_events),
    })

policy_comparison = pd.DataFrame(comparison_rows)
policy_comparison.to_csv(REPORT_DIR / "02_event_policy_comparison.csv", index=False)
policy_comparison

## 3. Time-gap check for each policy

TiSASRec needs meaningful time gaps. This table checks the gaps between consecutive interactions for each visitor after the simple five-event user minimum. It does not create training examples.

In [ ]:
gap_rows = []
for policy_name, allowed_events in POLICIES.items():
    policy_events = events.loc[events["event"].isin(allowed_events)].copy()
    active_users = policy_events.groupby("visitorid").size().loc[lambda counts: counts >= MIN_EVENTS_PER_USER].index
    sequence_events = policy_events.loc[policy_events["visitorid"].isin(active_users)].sort_values(
        ["visitorid", "timestamp"], kind="stable"
    )
    gaps_hours = sequence_events.groupby("visitorid")["timestamp"].diff().dropna() / 3_600_000
    gap_rows.append({
        "policy": policy_name,
        "gap_count": len(gaps_hours),
        "median_gap_hours": gaps_hours.median(),
        "p90_gap_hours": gaps_hours.quantile(0.90),
        "p99_gap_hours": gaps_hours.quantile(0.99),
        "distinct_gap_values": gaps_hours.nunique(),
    })

gap_summary = pd.DataFrame(gap_rows)
gap_summary.to_csv(REPORT_DIR / "03_time_gap_by_policy.csv", index=False)
gap_summary

## How to decide

Open `02_event_policy_comparison.csv` and `03_time_gap_by_policy.csv` in Excel. We will choose a policy using these questions:

1. Does it leave enough visitors with at least five interactions?
2. Does it retain enough items and events for a meaningful ranking task?
3. Are time gaps varied enough to test the time-aware model?
4. Does the policy match the question? Our question is next **eligible interaction**, not only next purchase.

For the current project, `all_events` is the starting policy. We should change it only if the report gives a strong reason. Record the final choice in `DECISION_LOG.md` before proceeding.